# Feature Extraction - Run Once, Use Everywhere

This notebook extracts all features (physiology, behavior, gaze) from preprocessing results and saves them to a pickle file.

**Parameterized**: Set `TIMEFRAME` to 'PRE' or 'POST' for pre-decision or post-decision analysis.

**Run this notebook ONCE after preprocessing completes.**

All other model notebooks will load the saved features instead of re-extracting.

In [12]:
# ============================================================================
# CONFIGURATION: Set timeframe for analysis
# ============================================================================
TIMEFRAME = 'POST'  # Options: 'PRE', 'POST' 
# ============================================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set time window based on TIMEFRAME
if TIMEFRAME == 'PRE':
    TIME_WINDOW = (-2.0, 0.0)  # PRE-decision: -2 to 0 seconds before submit
    SUFFIX = '_pre'
elif TIMEFRAME == 'POST':
    TIME_WINDOW = (0.0, 2.0)   # POST-decision: 0 to 2 seconds after submit
    SUFFIX = '_post'
else:
    raise ValueError("TIMEFRAME must be 'PRE' or 'POST'")

print(f"\n{'='*70}")
print(f"FEATURE EXTRACTION: {TIMEFRAME}-DECISION PERIOD")
print(f"Time window: {TIME_WINDOW[0]} to {TIME_WINDOW[1]} seconds")
print(f"{'='*70}\n")
print(f"Feature extraction started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


FEATURE EXTRACTION: POST-DECISION PERIOD
Time window: 0.0 to 2.0 seconds

Feature extraction started: 2026-03-28 17:20:27


## 1. Extract Physiology, Behavior, and Gaze Features

In [13]:
preprocessing_dir = Path('../../data/results/preprocessing')
preprocessing_files = sorted(preprocessing_dir.glob('preprocessing_*.json'))
raw_dir = Path('../../data/json')
baseline_method = 't3_stable_pre_decision'

print(f"Found {len(preprocessing_files)} preprocessing files")
print(f"Baseline correction method: {baseline_method}")

Found 105 preprocessing files
Baseline correction method: t3_stable_pre_decision


In [14]:
# =============================================================================
# UNIFIED FEATURE EXTRACTION: Physiology, Behavior, AND Gaze in ONE loop
# =============================================================================
# This ensures all modalities use the SAME trial filtering criteria

def extract_gaze_features(eye_data, submit_time, time_window):
    """
    Extract gaze features from raw eye tracking data for a specific time window.
    
    Parameters
    ----------
    eye_data : list
        Raw eye tracking samples from trial
    submit_time : float
        Timestamp of submit button press (milliseconds)
    time_window : tuple
        (start, end) in seconds relative to submit_time
    
    Returns
    -------
    dict or None
        Dictionary of gaze features, or None if insufficient data
    """
    if not eye_data or len(eye_data) == 0:
        return None
    
    timestamps = np.array([s['time'] for s in eye_data])
    
    # Filter to time window (convert window to milliseconds for comparison)
    time_relative_ms = timestamps - submit_time
    time_relative_s = time_relative_ms / 1000.0  # Convert to seconds
    
    time_mask = (time_relative_s >= time_window[0]) & (time_relative_s < time_window[1])
    indices = np.where(time_mask)[0]
    
    if len(indices) < 5:  # Need minimum samples
        return None
    
    # Apply filter
    eye_data_filtered = [eye_data[i] for i in indices]
    timestamps_filtered = timestamps[indices]
    
    # Extract gaze coordinates
    gaze_x_L = np.array([s.get('gazeL_X', np.nan) for s in eye_data_filtered])
    gaze_y_L = np.array([s.get('gazeL_Y', np.nan) for s in eye_data_filtered])
    gaze_x_R = np.array([s.get('gazeR_X', np.nan) for s in eye_data_filtered])
    gaze_y_R = np.array([s.get('gazeR_Y', np.nan) for s in eye_data_filtered])
    
    gaze_x = np.nanmean([gaze_x_L, gaze_x_R], axis=0)
    gaze_y = np.nanmean([gaze_y_L, gaze_y_R], axis=0)
    
    screen_x_L = np.array([s.get('pupilLSensorPosL_X', np.nan) for s in eye_data_filtered])
    screen_y_L = np.array([s.get('pupilLSensorPosL_Y', np.nan) for s in eye_data_filtered])
    screen_x_R = np.array([s.get('pupilLSensorPosR_X', np.nan) for s in eye_data_filtered])
    screen_y_R = np.array([s.get('pupilLSensorPosR_Y', np.nan) for s in eye_data_filtered])
    
    screen_x = np.nanmean([screen_x_L, screen_x_R], axis=0)
    screen_y = np.nanmean([screen_y_L, screen_y_R], axis=0)
    
    valid_L = np.array([s.get('validL', 0) for s in eye_data_filtered])
    valid_R = np.array([s.get('validR', 0) for s in eye_data_filtered])
    
    valid_mask = (valid_L > 0) & (valid_R > 0)
    if valid_mask.sum() < 5:
        return None
    
    # Extract valid samples only
    gaze_x_valid = gaze_x[valid_mask]
    gaze_y_valid = gaze_y[valid_mask]
    screen_x_valid = screen_x[valid_mask]
    screen_y_valid = screen_y[valid_mask]
    timestamps_valid = timestamps_filtered[valid_mask]
    
    # Calculate features
    features = {}
    features['gaze_valid_pct'] = np.mean(valid_mask)
    features['gaze_x_mean'] = np.nanmean(gaze_x_valid)
    features['gaze_x_std'] = np.nanstd(gaze_x_valid)
    features['gaze_y_mean'] = np.nanmean(gaze_y_valid)
    features['gaze_y_std'] = np.nanstd(gaze_y_valid)
    features['screen_x_mean'] = np.nanmean(screen_x_valid)
    features['screen_x_std'] = np.nanstd(screen_x_valid)
    features['screen_y_mean'] = np.nanmean(screen_y_valid)
    features['screen_y_std'] = np.nanstd(screen_y_valid)
    
    # Velocity and acceleration
    dt = np.diff(timestamps_valid)
    dt[dt == 0] = 1e-6
    dx = np.diff(screen_x_valid)
    dy = np.diff(screen_y_valid)
    
    velocity = np.sqrt(dx**2 + dy**2) / dt
    features['gaze_velocity_mean'] = np.nanmean(velocity)
    features['gaze_velocity_std'] = np.nanstd(velocity)
    features['gaze_velocity_max'] = np.nanmax(velocity) if len(velocity) > 0 else 0
    
    if len(velocity) > 1:
        acceleration = np.diff(velocity) / dt[:-1]
        features['gaze_acceleration_mean'] = np.nanmean(np.abs(acceleration))
        features['gaze_acceleration_std'] = np.nanstd(acceleration)
    else:
        features['gaze_acceleration_mean'] = 0
        features['gaze_acceleration_std'] = 0
    
    # Fixation and saccade metrics
    fixation_mask = velocity < 30
    saccade_mask = velocity > 100
    features['fixation_ratio'] = np.mean(fixation_mask) if len(fixation_mask) > 0 else 0
    features['saccade_ratio'] = np.mean(saccade_mask) if len(saccade_mask) > 0 else 0
    features['saccade_count'] = np.sum(np.diff(saccade_mask.astype(int)) == 1) if len(saccade_mask) > 1 else 0
    
    # Dispersion and path length
    features['gaze_dispersion_x'] = np.nanmax(screen_x_valid) - np.nanmin(screen_x_valid)
    features['gaze_dispersion_y'] = np.nanmax(screen_y_valid) - np.nanmin(screen_y_valid)
    features['gaze_path_length'] = np.sum(np.sqrt(dx**2 + dy**2))
    
    return features


# =============================================================================
# MAIN EXTRACTION LOOP - All modalities extracted together
# =============================================================================
all_features = []  # Single list for all trial data
total_trials = 0
skipped_no_gaze = 0

for preprocessed_file in preprocessing_files:
    with open(preprocessed_file, 'r') as f:
        preprocessed = json.load(f)
    
    subject_id = preprocessed['subject_id']
    print(f"\nProcessing subject: {subject_id}", end=" ")
    
    # Find matching raw JSON file
    matches = list(raw_dir.glob(f"*{subject_id.split('_')[-1]}.json"))
    pattern = subject_id.replace("_", ".*")
    match = next((f for f in matches if re.search(pattern, f.name)), None)
    if not match:
        print("❌ No matching JSON file")
        continue
    
    with open(match, 'r') as f:
        raw_data = json.load(f)
    
    subject_trial_count = 0
    subject_skipped_gaze = 0
    
    for trial_id, trial_data in preprocessed['trial_data'].items():
        method_data = trial_data['methods'][baseline_method]
        
        # Filter 1: Preprocessing must have succeeded
        if method_data['success'] != True:
            continue
        
        raw_trial = raw_data['trials'][int(trial_id)-1]
        
        # Filter 2: Trial must be submitted
        if not raw_trial['gamble details']['submitted']:
            continue
        
        # =====================================================================
        # PHYSIOLOGY FEATURES
        # =====================================================================
        time_aligned = np.array(trial_data['time_relative_to_submit'])
        pupil_avg = np.array(method_data['pupil_avg_baselined'])
        pupil_L = np.array(method_data['pupil_L_baselined'])
        pupil_R = np.array(method_data['pupil_R_baselined'])

        valid_mask = ~np.isnan(pupil_avg)
        pupil_avg_clean = pupil_avg[valid_mask]
        pupil_L_clean = pupil_L[valid_mask]
        pupil_R_clean = pupil_R[valid_mask]
        time_clean = time_aligned[valid_mask]

        # Filter 3: Need enough valid pupil samples overall
        if len(pupil_avg_clean) < 20:
            continue

        # Filter to time window (PRE or POST)
        if TIMEFRAME == 'PRE':
            time_mask = (time_clean >= TIME_WINDOW[0]) & (time_clean < TIME_WINDOW[1])
        else:  # POST
            time_mask = (time_clean > TIME_WINDOW[0]) & (time_clean <= TIME_WINDOW[1])
        
        pupil = pupil_avg_clean[time_mask]
        pupil_L_filtered = pupil_L_clean[time_mask]
        pupil_R_filtered = pupil_R_clean[time_mask]
        time_filtered = time_clean[time_mask]

        # Filter 4: Need enough pupil samples in time window
        if len(pupil) < 5:
            continue

        # Calculate derivatives
        pupil_velocity = np.diff(pupil) if len(pupil) > 1 else np.array([0])
        pupil_acceleration = np.diff(pupil_velocity) if len(pupil_velocity) > 1 else np.array([0])
        dilation_mask = pupil_velocity > 0 if len(pupil_velocity) > 0 else np.array([False])

        # Physiology features
        physio_features = {
            f'pupil_mean{SUFFIX}': np.mean(pupil),
            f'pupil_std{SUFFIX}': np.std(pupil),
            f'pupil_slope{SUFFIX}': np.polyfit(time_filtered, pupil, 1)[0] if len(time_filtered) > 1 else 0,
            f'time_to_peak{SUFFIX}': time_filtered[np.argmax(pupil)] - time_filtered[0] if len(time_filtered) > 0 else 0,
            f'pupil_cv{SUFFIX}': np.std(pupil) / np.abs(np.mean(pupil)) if (len(pupil) > 0 and np.mean(pupil) != 0) else 0,
            f'pupil_velocity_mean{SUFFIX}': np.mean(np.abs(pupil_velocity)) if len(pupil_velocity) > 0 else 0,
            f'pupil_max_dilation_rate{SUFFIX}': np.max(pupil_velocity) if len(pupil_velocity) > 0 else 0,
            f'pupil_max_constriction_rate{SUFFIX}': np.abs(np.min(pupil_velocity)) if len(pupil_velocity) > 0 else 0,
            f'pupil_acceleration_std{SUFFIX}': np.std(pupil_acceleration) if len(pupil_acceleration) > 1 else 0,
            f'pct_time_dilating{SUFFIX}': np.mean(dilation_mask) if len(dilation_mask) > 0 else 0,
            f'num_dilation_peaks{SUFFIX}': np.sum(np.diff(np.sign(pupil_velocity)) > 0) if len(pupil_velocity) > 1 else 0,
            f'eye_asymmetry{SUFFIX}': np.nanmean(np.abs(pupil_L_filtered - pupil_R_filtered)) if len(pupil_L_filtered) > 0 else 0,
            f'eye_asymmetry_std{SUFFIX}': np.nanstd(pupil_L_filtered - pupil_R_filtered) if len(pupil_L_filtered) > 1 else 0,
        }
        
        # =====================================================================
        # BEHAVIOR FEATURES
        # =====================================================================
        gamble_params = raw_trial['gamble details']['gamble parameters']
        lct = raw_trial['lct']
        
        show_screen_time = None
        submit_time = None
        click_time = None
        
        for event in lct:
            if 'show screen' in event['event']:
                show_screen_time = event['time']
            elif 'gamble clicked' in event['event']:
                click_time = event['time']
            elif 'submit' in event['event']:
                submit_time = event['time']
        
        # Filter 5: Must have timing data
        if show_screen_time is None or submit_time is None:
            continue
        
        decision_time = (submit_time - show_screen_time)

        # Expected values
        invest_amount_1 = gamble_params['invest amount 1']
        invest_amount_2 = gamble_params['invest amount 2']
        invest_prob_1 = gamble_params['invest probability 1']
        invest_prob_2 = gamble_params['invest probability 2']
        keep_amount = gamble_params['keep amount']
        
        invest_ev = invest_amount_1 * invest_prob_1 + invest_amount_2 * invest_prob_2
        ev_difference = invest_ev - keep_amount
        invest_variance = ((invest_amount_1 - invest_ev)**2 * invest_prob_1 +
                          (invest_amount_2 - invest_ev)**2 * invest_prob_2)
        
        final_choice = raw_trial['gamble details']['choices'][-1]['choice'] if len(raw_trial['gamble details']['choices']) > 0 else None
        outcome = 1 if final_choice == 'INVEST' else 0
        
        behavior_features = {
            'decision_time': decision_time,
            'ev_difference': ev_difference,
            'invest_variance': invest_variance,
            'ambiguity': gamble_params['ambiguity'],
            'condition_social': 1 if gamble_params['condition'] == 'social' else 0,
            'risk_premium': ev_difference / np.sqrt(invest_variance) if invest_variance > 0 else 0,
        }
        
        # =====================================================================
        # GAZE FEATURES (extracted in same loop with same filtering)
        # =====================================================================
        eye_data = raw_trial.get('eye', [])
        gaze_features = extract_gaze_features(eye_data, submit_time, TIME_WINDOW)
        
        # Filter 6: Must have valid gaze data (consistent with pupil filtering)
        if gaze_features is None:
            subject_skipped_gaze += 1
            continue
        
        # =====================================================================
        # COMBINE ALL FEATURES FOR THIS TRIAL
        # =====================================================================
        trial_features = {
            'subject_id': subject_id,
            'trial_id': f"{trial_id}_{subject_id}",
            'trial_number': int(trial_id),
            'outcome': outcome,
            **physio_features,
            **behavior_features,
            **gaze_features
        }
        
        all_features.append(trial_features)
        subject_trial_count += 1
    
    print(f"✓ {subject_trial_count} trials (skipped {subject_skipped_gaze} for gaze)")
    total_trials += subject_trial_count
    skipped_no_gaze += subject_skipped_gaze

print(f"\n{'='*80}")
print(f"EXTRACTION COMPLETE")
print(f"{'='*80}")
print(f"Total trials extracted: {total_trials}")
print(f"Total skipped (no valid gaze): {skipped_no_gaze}")
print(f"\n✓ All modalities extracted with CONSISTENT filtering")


Processing subject: 0727_1400_539136F ✓ 0 trials (skipped 0 for gaze)

Processing subject: 0727_1400_A6I5HI6 ✓ 0 trials (skipped 0 for gaze)

Processing subject: 0731_1000_539136F ✓ 0 trials (skipped 117 for gaze)

Processing subject: 0731_1000_A6I5HI6 ✓ 0 trials (skipped 124 for gaze)

Processing subject: 0731_1000_U9TEJGM ✓ 0 trials (skipped 126 for gaze)

Processing subject: 0802_1400_539136F ✓ 0 trials (skipped 129 for gaze)

Processing subject: 0802_1400_A6I5HI6 ✓ 0 trials (skipped 131 for gaze)

Processing subject: 0802_1400_U9TEJGM ✓ 0 trials (skipped 131 for gaze)

Processing subject: 0806_1000_U9TEJGM ✓ 0 trials (skipped 131 for gaze)

Processing subject: 0811_1000_4LI8GO7 ✓ 0 trials (skipped 121 for gaze)

Processing subject: 0811_1000_539136F ✓ 0 trials (skipped 130 for gaze)

Processing subject: 0811_1000_U9TEJGM ✓ 0 trials (skipped 131 for gaze)

Processing subject: 0813_1000_539136F ✓ 0 trials (skipped 129 for gaze)

Processing subject: 0813_1000_9M4VCHG ✓ 0 trials (skip

## 2. Create DataFrames and Merge

In [15]:
# =============================================================================
# CREATE DATAFRAME AND ADD SEQUENTIAL FEATURES
# =============================================================================
# All features are already in a single list - just convert to DataFrame

merged_df = pd.DataFrame(all_features)

print(f"Created DataFrame with {len(merged_df)} trials")
print(f"Columns: {len(merged_df.columns)}")

# Debug: show all columns to verify gaze features are present
print(f"\nAll columns in DataFrame:")
print(merged_df.columns.tolist())

# ==============================================================================
# ADD PERCENTAGE-BASED / SEQUENTIAL FEATURES
# ==============================================================================
print("\nAdding percentage-based sequential features...")

# Sort by subject and trial number to ensure correct ordering
merged_df = merged_df.sort_values(['subject_id', 'trial_number']).reset_index(drop=True)

# Group by subject for sequential calculations
grouped = merged_df.groupby('subject_id')

# 1. Running investment rate (cumulative % of INVEST choices up to but not including current trial)
merged_df['running_invest_rate'] = grouped['outcome'].transform(
    lambda x: x.expanding().mean().shift(1)
)
merged_df['running_invest_rate'] = merged_df['running_invest_rate'].fillna(0.5)  # Default to 50% for first trial

# 2. Recent investment rate (last 5 trials)
merged_df['recent_invest_rate_5'] = grouped['outcome'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
merged_df['recent_invest_rate_5'] = merged_df['recent_invest_rate_5'].fillna(0.5)

# 3. Previous trial outcome (lag-1)
merged_df['prev_outcome'] = grouped['outcome'].shift(1)
merged_df['prev_outcome'] = merged_df['prev_outcome'].fillna(0.5)  # Default for first trial

# 4. Trial position within session (normalized 0-1)
merged_df['trial_position'] = grouped['trial_number'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0.5
)

# 5. Consecutive same decisions counter
def count_consecutive(series):
    """Count consecutive identical values, resetting on change."""
    result = []
    count = 0
    prev_val = None
    for val in series:
        if prev_val is None:
            count = 0
        elif val == prev_val:
            count += 1
        else:
            count = 0
        result.append(count)
        prev_val = val
    return result

merged_df['consecutive_same'] = grouped['outcome'].transform(
    lambda x: pd.Series(count_consecutive(x.shift(1).fillna(-1).values), index=x.index)
)

print(f"  Added: running_invest_rate, recent_invest_rate_5, prev_outcome, trial_position, consecutive_same")

# ==============================================================================
# IDENTIFY FEATURE COLUMNS
# ==============================================================================
# Physiology: columns ending with _pre or _post suffix
physio_cols = [c for c in merged_df.columns if c.endswith('_pre') or c.endswith('_post')]

# Gaze: columns starting with gaze_ or screen_, plus specific gaze metrics
gaze_cols = [c for c in merged_df.columns 
             if c.startswith('gaze_') or c.startswith('screen_') or 
             c in ['fixation_ratio', 'saccade_ratio', 'saccade_count']]

# Behavior: core + sequential features
behavior_cols = [
    'decision_time', 'ev_difference', 'invest_variance', 
    'ambiguity', 'condition_social', 'risk_premium',
    'running_invest_rate', 'recent_invest_rate_5', 'prev_outcome', 
    'trial_position', 'consecutive_same'
]

# Debug: print identified columns
print(f"\nIdentified physio columns ({len(physio_cols)}):")
print(physio_cols)
print(f"\nIdentified gaze columns ({len(gaze_cols)}):")
print(gaze_cols)

print(f"\n{'='*80}")
print(f"FINAL DATASET ({TIMEFRAME}-DECISION):")
print(f"{'='*80}")
print(f"Total trials: {len(merged_df)}")
print(f"Subjects: {merged_df['subject_id'].nunique()}")
print(f"\nFeature counts:")
print(f"  Physiology: {len(physio_cols)} features")
print(f"  Behavior: {len(behavior_cols)} features (6 core + 5 sequential)")
print(f"  Gaze: {len(gaze_cols)} features")
print(f"\nOutcome distribution:")
print(merged_df['outcome'].value_counts())
print(f"\n✓ All modalities extracted with CONSISTENT trial filtering (no imputation needed)")

Created DataFrame with 0 trials
Columns: 0

All columns in DataFrame:
[]

Adding percentage-based sequential features...


KeyError: 'subject_id'

## 3. Save to Pickle File

In [ ]:
# =============================================================================
# SAVE TO PICKLE FILE
# =============================================================================
output_dir = Path(f'../../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f'extracted_features_{TIMEFRAME}.pkl'

# Save with metadata
feature_data = {
    'merged_df': merged_df,
    'physio_cols': physio_cols,
    'behavior_cols': behavior_cols,
    'gaze_cols': gaze_cols,
    'metadata': {
        'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'n_trials': len(merged_df),
        'n_subjects': merged_df['subject_id'].nunique(),
        'baseline_method': baseline_method,
        'preprocessing_files': len(preprocessing_files),
        'time_window': f'{TIMEFRAME}-decision ({TIME_WINDOW[0]} to {TIME_WINDOW[1]} seconds)',
        'extraction_method': 'unified_loop',  # All modalities extracted together
        'filtering': {
            'preprocessing_success': True,
            'trial_submitted': True,
            'min_pupil_samples_overall': 20,
            'min_pupil_samples_in_window': 5,
            'min_gaze_samples_in_window': 5,
            'min_valid_gaze_samples': 5,
        },
        'behavior_features': {
            'core': ['decision_time', 'ev_difference', 'invest_variance', 
                     'ambiguity', 'condition_social', 'risk_premium'],
            'sequential': ['running_invest_rate', 'recent_invest_rate_5', 'prev_outcome', 
                          'trial_position', 'consecutive_same']
        },
        'notes': 'All modalities extracted in single loop with consistent filtering. No imputation needed.'
    }
}

with open(output_file, 'wb') as f:
    pickle.dump(feature_data, f)

print(f"\n{'='*80}")
print(f"✓ {TIMEFRAME}-decision features saved to: {output_file}")
print(f"{'='*80}")
print(f"\nSaved data includes:")
print(f"  - merged_df: {len(merged_df)} trials × {len(merged_df.columns)} columns")
print(f"  - physio_cols: {len(physio_cols)} features")
print(f"  - behavior_cols: {len(behavior_cols)} features")
print(f"  - gaze_cols: {len(gaze_cols)} features")
print(f"  - metadata: extraction details and filtering criteria")


✓ PRE-decision features saved to: ../../data/features/extracted_features_PRE.pkl

Saved data includes:
  - merged_df: 12107 trials × 48 columns
  - physio_cols: 13 features
  - behavior_cols: 11 features
  - gaze_cols: 20 features
  - metadata: extraction details and filtering criteria
